# XGBoost Traffic Model Benchmark
This notebook trains an XGBoost model using the shared data pipeline.

In [ ]:
import xgboost as xgb
import optuna
import matplotlib.pyplot as plt
import model_utils

# 1. Load Data
data, features, cat_cols, target = model_utils.load_traffic_data()


data["congestion"].round(1).value_counts()

In [ ]:
# 2. Split Data
X_train, X_test, y_train, y_test = model_utils.get_train_test_split(data, features, target)

# Convert to DMatrix for GPU training
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)


In [ ]:
# 3. Hyperparameter Tuning with Optuna
import os
import hashlib
import json

PARAM_CACHE_FILE = "../models/xgboost_best_params.json"

def objective(trial):
    """Optuna objective function for XGBoost hyperparameter tuning"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 6, 15),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'tree_method': 'hist',
        'device': 'cuda',
        'random_state': 42
    }
    
    # Train using native XGBoost API with DMatrix for GPU
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=params['n_estimators'],
        evals=[(dtest, 'test')],
        verbose_eval=False
    )
    
    preds = model.predict(dtest)
    mae = model_utils.mean_absolute_error(y_test, preds)
    
    return mae

# Run Optuna study
print("Starting hyperparameter optimization on GPU...")
study = optuna.create_study(direction='minimize', study_name='xgboost_traffic')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print(f"\nBest MAE: {study.best_value:.6f}")
print(f"Best parameters: {study.best_params}")

# Save best params to file
best_params = study.best_params.copy()
best_params['tree_method'] = 'hist'
best_params['device'] = 'cuda'
best_params['random_state'] = 42

with open(PARAM_CACHE_FILE, 'w') as f:
    json.dump(best_params, f, indent=2)
print(f"\nBest parameters saved to {PARAM_CACHE_FILE}")


In [ ]:
# 4. Train Final Model with Best Parameters
import json

PARAM_CACHE_FILE = "../models/xgboost_best_params.json"

# Load best params from cache
if os.path.exists(PARAM_CACHE_FILE):
    print(f"Loading parameters from {PARAM_CACHE_FILE}...")
    with open(PARAM_CACHE_FILE, 'r') as f:
        params = json.load(f)
else:
    raise FileNotFoundError(f"{PARAM_CACHE_FILE} not found. Run the tuning cell first.")

# Generate unique run ID based on params
param_str = json.dumps(params, sort_keys=True)
run_hash = hashlib.md5(param_str.encode("utf-8")).hexdigest()[:8]
model_path = f"../models/../models/traffic_xgb_{run_hash}.json"

# Train using native XGBoost API with DMatrix for GPU
if os.path.exists(model_path):
    print(f"Loading cached model from {model_path}...")
    model = xgb.Booster()
    model.load_model(model_path)
else:
    print(f"Training new model on GPU (Hash: {run_hash})...")
    num_boost_round = params.pop('n_estimators', 1000)
    
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtest, 'test')],
        verbose_eval=100
    )
    model.save_model(model_path)
    print(f"Model saved to {model_path}")


In [ ]:
# 5. Evaluate
preds = model.predict(dtest)
print(f"MAE: {model_utils.mean_absolute_error(y_test, preds):.6f}")
print(f"R²: {model_utils.r2_score(y_test, preds):.6f}")

# For plotting, we need to create a sklearn-compatible wrapper
class XGBWrapper:
    def __init__(self, booster):
        self.booster = booster
    
    def predict(self, X):
        dmatrix = xgb.DMatrix(X, enable_categorical=True)
        return self.booster.predict(dmatrix)

wrapper = XGBWrapper(model)
model_utils.plot_benchmark_profile(wrapper, features, "XGBoost")


In [ ]:
# 6. Feature Importance
fig, ax = plt.subplots(figsize=(10, 6))
xgb.plot_importance(model, ax=ax, max_num_features=15, height=0.5)
plt.show()
